In [1]:
# Imports, Settings, and Function Definitions

import os
import numpy as np
import torch
import pandas as pd
from glob import glob
from scipy.io import loadmat
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, periodogram
from sklearn.preprocessing import StandardScaler
from scipy.stats import kurtosis
from torch.utils.data import random_split
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import gc
from sklearn.metrics import r2_score
from scipy.io import savemat
import joblib
from scipy.signal import hilbert
from sklearn.decomposition import PCA
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from typing import Literal

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

2025-05-29 16:06:46.825314: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-29 16:06:46.833788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748549206.843806  211038 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748549206.846787  211038 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748549206.854622  211038 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Using Device: cuda


# Creating a Multi-Task Learning Model

Best Model Architecture: Multi-Task Learning (MTL)

After evaluating your requirements—real-time performance, applicability to unilateral and bilateral movements, and alignment with natural cognitive processes—I recommend a Multi-Task Learning (MTL) approach with a Shared Feature Extractor and Task-Specific Heads. This architecture allows one model to handle all scenarios efficiently while leveraging shared knowledge across tasks.
Why Multi-Task Learning?

    Shared Knowledge: A shared feature extractor learns general patterns from ECoG data applicable to both wrists, improving generalization.
    Efficiency: One model reduces computational overhead compared to maintaining separate models or a complex MOE setup.
    Flexibility: It naturally predicts movements for both wrists, handling unilateral cases (by ignoring one output) and bilateral cases (using both outputs).
    Biological Alignment: The shared extractor mimics how the motor cortex processes raw signals, with separate pathways decoding specific movements—closer to natural cognition than a gated MOE.

Model Architecture

    Shared Feature Extractor:
        A deep neural network (e.g., Convolutional Neural Network [CNN] for spatial features or Recurrent Neural Network [RNN] for temporal sequences) processes the preprocessed ECoG data.
        Outputs a feature representation capturing patterns common to both wrists.
    Task-Specific Heads:
        Left Wrist Head: A small network (e.g., feedforward layer) predicting left wrist movement.
        Right Wrist Head: A similar network predicting right wrist movement.
        Each head takes the shared features as input and outputs movement predictions (e.g., 0 for no movement, 1 for movement).
    Output Handling:
        Unilateral Movement: Predict both outputs; use only the relevant wrist’s prediction (e.g., mask the other as 0).
        Bilateral Movement: Use both outputs to predict the sequence (e.g., right wrist moves, then left).
    Loss Function:
        Combine losses from both tasks (e.g., sum or weighted sum of binary cross-entropy losses) to train the model end-to-end.

# Step 1: Data Preparation

Dataset: Use your bilateral movement dataset, ensuring labels indicate movements for both wrists (e.g., [left: 0, right: 1] for right-only, [left: 1, right: 1] for bilateral).

Unilateral Cases: For left-only or right-only data, label the inactive wrist as 0.

Preprocessing: Reuse your existing preprocessing (band extraction, feature extraction) to maintain consistency.

In [11]:
# Preprocessing Data

# Define Training and Test data
# Read in the data
movement_direction = "Bilateral"

processed_data_l_X = sorted(glob(os.path.join('/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/', movement_direction, "**", "X.npy")))
processed_data_l_y = sorted(glob(os.path.join('/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/', movement_direction, "**", "y.npy")))

In [12]:
dir = '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data'
dir

'/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data'

In [13]:
movement_direction

'Bilateral'

In [23]:
motion_data_l = sorted(glob(os.path.join(dir, movement_direction, "**", "motion_data.csv")))
ecog_data = sorted(glob(os.path.join(dir, movement_direction, "**", "ecog_data.csv")))

In [24]:
ecog_data

['/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-07-12_(S1)/ecog_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-07-19_(S2)/ecog_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-07-26_(S3)/ecog_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-08-14_(S4)/ecog_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-09-27_(S5)/ecog_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-10-04_(S6)/ecog_data.csv']

In [25]:
motion_data_l

['/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-07-12_(S1)/motion_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-07-19_(S2)/motion_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-07-26_(S3)/motion_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-08-14_(S4)/motion_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-09-27_(S5)/motion_data.csv',
 '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Bilateral/2018-10-04_(S6)/motion_data.csv']

In [ ]:
processed_data_l_y 

[]